In [1]:
pip install catboost

  Using cached catboost-1.2.10-cp313-cp313-win_amd64.whl.metadata (1.5 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
Using cached catboost-1.2.10-cp313-cp313-win_amd64.whl (100.2 MB)

   ---------------------------------------- 0/2 [graphviz]
   -------------------- ------------------- 1/2 [catboost]
   -------------------- ------------------- 1/2 [catboost]
   -------------------- ------------------- 1/2 [catboost]
   -------------------- ------------------- 1/2 [catboost]
   -------------------- ------------------- 1/2 [catboost]
   -------------------- ------------------- 1/2 [catboost]
   -------------------- ------------------- 1/2 [catboost]
   -------------------- ------------------- 1/2 [catboost]
   -------------------- ------------------- 1/2 [catboost]
   -------------------- ------------------- 1/2 [catboost]
   -------------------- ------------------- 1/2 [catboost]
   -------------------- ------------------- 1/2 [catboost]
   -------------------- ----

# обработка

In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool, cv
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv(
    "realty_clean.csv", 
    encoding='utf-8-sig',
    sep=';',
    on_bad_lines='skip',
    low_memory=False
)

In [60]:
import pandas as pd

df = pd.read_csv(
    "realty_clean.csv", 
    encoding='utf-8-sig',
    sep=';',
    on_bad_lines='skip',
    low_memory=False
)

print(f"Загружено {len(df)} строк")

Загружено 1642 строк


In [61]:
print(f" Колонки: {list(df.columns)}")

 Колонки: ['offer_id', 'title', 'price', 'total_area', 'area_living', 'area_kitchen', 'rooms', 'floor', 'total_floors', 'address', 'description', 'ceiling_height', 'build_year', 'house_type', 'bathroom', 'renovation', 'url', 'images_count', 's3_uris_count', 's3_uris', 'timestamp,title,price,total_area,rooms,floor,total_floors,address,latitude,longitude,description,json_description,url,images_count', 'price_per_m2', 'floor_category', 'apartment_type', 'desc_length', 'has_renovation', 'has_parking', 'has_balcony']


In [62]:
len(list(df.columns))

28

In [63]:
cols_to_drop = [col for col in df.columns if 'timestamp' in col or 'json_description' in col or 'latitude' in col or 'longitude' in col]
df = df.drop(columns=cols_to_drop, errors='ignore')

In [64]:
len(list(df.columns))

27

In [65]:
list(df.columns)

['offer_id',
 'title',
 'price',
 'total_area',
 'area_living',
 'area_kitchen',
 'rooms',
 'floor',
 'total_floors',
 'address',
 'description',
 'ceiling_height',
 'build_year',
 'house_type',
 'bathroom',
 'renovation',
 'url',
 'images_count',
 's3_uris_count',
 's3_uris',
 'price_per_m2',
 'floor_category',
 'apartment_type',
 'desc_length',
 'has_renovation',
 'has_parking',
 'has_balcony']

In [66]:
df.head()

,offer_id,title,price,total_area,area_living,area_kitchen,rooms,floor,total_floors,address,...,images_count,s3_uris_count,s3_uris,price_per_m2,floor_category,apartment_type,desc_length,has_renovation,has_parking,has_balcony
0,3190558370547190502,"37,9 м², квартира-студия",30445648,37.9,15.6,NaN,студия,29.0,35.0,"Москва, ЖК Родина Парк, к1, ЖК «Родина Парк»",...,15.0,15.0,s3://realty-images/offers/3190558370547190502/...,8.033153e+05,high,studio,10293,1,1,0
1,5180127926972929670,"66,4 м², 3-комнатная квартира",35000000,66.4,15.0,NaN,3,16.0,16.0,"Москва, Небесный бульвар, 1к1, Жилой район «Алиа»",...,15.0,15.0,s3://realty-images/offers/5180127926972929670/...,5.271084e+05,high,medium,10219,1,1,0
2,2159627191506509600,"71,5 м², 2-комнатная квартира",102118256,71.5,NaN,NaN,2,14.0,18.0,"Москва, Украинский бульвар, 2с1, ЖК «Бадаевский»",...,15.0,15.0,s3://realty-images/offers/2159627191506509600/...,1.428227e+06,high,small,9853,1,1,0
3,5228459389428247141,"138 м², 4-комнатная квартира",135000000,138.0,NaN,NaN,4,7.0,9.0,"Москва, Малая Бронная улица, 25, ✅ Характерист...",...,15.0,15.0,s3://realty-images/offers/5228459389428247141/...,9.782609e+05,high,medium,12273,1,1,1
4,7035113340557147293,"74,4 м², 3-комнатная квартира",22145000,74.4,10.2,10.0,3,2.0,17.0,"Москва, улица Барышиха, 50, ✅ Характеристики, ...",...,15.0,15.0,s3://realty-images/offers/7035113340557147293/...,2.976478e+05,low,medium,13622,1,1,1


In [67]:
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [68]:
categorical_cols = ['rooms', 'house_type', 'bathroom', 'renovation', 'floor_category', 'apartment_type']

In [69]:
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype(str)
        df[col] = df[col].replace('nan', pd.NA)
        df[col] = df[col].fillna('unknown')

In [70]:
df.head()

,offer_id,title,price,total_area,area_living,area_kitchen,rooms,floor,total_floors,address,...,images_count,s3_uris_count,s3_uris,price_per_m2,floor_category,apartment_type,desc_length,has_renovation,has_parking,has_balcony
0,3190558370547190502,"37,9 м², квартира-студия",30445648,37.9,15.6,NaN,студия,29.0,35.0,"Москва, ЖК Родина Парк, к1, ЖК «Родина Парк»",...,15.0,15.0,s3://realty-images/offers/3190558370547190502/...,8.033153e+05,high,studio,10293,1,1,0
1,5180127926972929670,"66,4 м², 3-комнатная квартира",35000000,66.4,15.0,NaN,3,16.0,16.0,"Москва, Небесный бульвар, 1к1, Жилой район «Алиа»",...,15.0,15.0,s3://realty-images/offers/5180127926972929670/...,5.271084e+05,high,medium,10219,1,1,0
2,2159627191506509600,"71,5 м², 2-комнатная квартира",102118256,71.5,NaN,NaN,2,14.0,18.0,"Москва, Украинский бульвар, 2с1, ЖК «Бадаевский»",...,15.0,15.0,s3://realty-images/offers/2159627191506509600/...,1.428227e+06,high,small,9853,1,1,0
3,5228459389428247141,"138 м², 4-комнатная квартира",135000000,138.0,NaN,NaN,4,7.0,9.0,"Москва, Малая Бронная улица, 25, ✅ Характерист...",...,15.0,15.0,s3://realty-images/offers/5228459389428247141/...,9.782609e+05,high,medium,12273,1,1,1
4,7035113340557147293,"74,4 м², 3-комнатная квартира",22145000,74.4,10.2,10.0,3,2.0,17.0,"Москва, улица Барышиха, 50, ✅ Характеристики, ...",...,15.0,15.0,s3://realty-images/offers/7035113340557147293/...,2.976478e+05,low,medium,13622,1,1,1


In [71]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1642 entries, 0 to 1641
Data columns (total 27 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   offer_id        1642 non-null   int64  
 1   title           1642 non-null   object 
 2   price           1642 non-null   int64  
 3   total_area      1642 non-null   float64
 4   area_living     1363 non-null   float64
 5   area_kitchen    24 non-null     float64
 6   rooms           1642 non-null   object 
 7   floor           1642 non-null   float64
 8   total_floors    1642 non-null   float64
 9   address         1642 non-null   object 
 10  description     1642 non-null   object 
 11  ceiling_height  625 non-null    float64
 12  build_year      1642 non-null   float64
 13  house_type      1642 non-null   object 
 14  bathroom        1642 non-null   object 
 15  renovation      1642 non-null   object 
 16  url             1642 non-null   object 
 17  images_count    1642 non-null   f

In [72]:
df = df.drop(columns=['area_kitchen'])

In [73]:
df['area_living_filled'] = df['area_living'].fillna(df['total_area'] * 0.7)
df['area_living'] = df['area_living_filled']
df = df.drop(columns=['area_living_filled'])

In [74]:
median_height = df['ceiling_height'].median()
df['ceiling_height'] = df['ceiling_height'].fillna(median_height)

In [75]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1642 entries, 0 to 1641
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   offer_id        1642 non-null   int64  
 1   title           1642 non-null   object 
 2   price           1642 non-null   int64  
 3   total_area      1642 non-null   float64
 4   area_living     1642 non-null   float64
 5   rooms           1642 non-null   object 
 6   floor           1642 non-null   float64
 7   total_floors    1642 non-null   float64
 8   address         1642 non-null   object 
 9   description     1642 non-null   object 
 10  ceiling_height  1642 non-null   float64
 11  build_year      1642 non-null   float64
 12  house_type      1642 non-null   object 
 13  bathroom        1642 non-null   object 
 14  renovation      1642 non-null   object 
 15  url             1642 non-null   object 
 16  images_count    1642 non-null   float64
 17  s3_uris_count   1642 non-null   f

In [76]:
total_missing = df.isna().sum().sum()
total_missing

np.int64(0)

In [117]:
target = 'price'

numeric_features = [
    'total_area', 
    'area_living', 
    'floor', 
    'total_floors', 
    'build_year',
    'ceiling_height',
    'desc_length',
    'has_renovation', 
    'has_parking', 
    'has_balcony'
]

categorical_features = [
    'rooms', 
    'house_type', 
    'bathroom', 
    'renovation', 
    'floor_category', 
    'apartment_type'
]

In [118]:
print(f"📊 Числовые признаки ({len(numeric_features)}):")
for f in numeric_features:
    print(f"   - {f}")

print(f"\n📊 Категориальные признаки ({len(categorical_features)}):")
for f in categorical_features:
    print(f"   - {f}")

📊 Числовые признаки (10):
   - total_area
   - area_living
   - floor
   - total_floors
   - build_year
   - ceiling_height
   - desc_length
   - has_renovation
   - has_parking
   - has_balcony

📊 Категориальные признаки (6):
   - rooms
   - house_type
   - bathroom
   - renovation
   - floor_category
   - apartment_type


In [119]:
X = df[numeric_features + categorical_features].copy()
y = df[target].copy()

y_log = np.log1p(y)

print(f" Размер X: {X.shape}")
print(f" Размер y: {len(y)}")
print(f"\nПример данных:")
X.head()

 Размер X: (1642, 16)
 Размер y: 1642

Пример данных:


,total_area,area_living,floor,total_floors,build_year,ceiling_height,desc_length,has_renovation,has_parking,has_balcony,rooms,house_type,bathroom,renovation,floor_category,apartment_type
0,37.9,15.60,29.0,35.0,2028.0,3.16,10293,1,1,0,студия,unknown,раздельный,unknown,high,studio
1,66.4,15.00,16.0,16.0,2024.0,3.25,10219,1,1,0,3,кирпичный,раздельный,премиум ремонт,high,medium
2,71.5,50.05,14.0,18.0,2026.0,3.25,9853,1,1,0,2,кирпичный,раздельный,без отделки,high,small
3,138.0,96.60,7.0,9.0,1998.0,3.25,12273,1,1,1,4,монолитный,совмещенный,без отделки,high,medium
4,74.4,10.20,2.0,17.0,2016.0,3.25,13622,1,1,1,3,монолитный,совмещенный,косметический ремонт,low,medium


In [120]:
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.3, random_state=42, shuffle=True)

print(f"📊 Обучающая выборка: {len(X_train)} записей ({len(X_train)/len(X)*100:.0f}%)")
print(f"📊 Тестовая выборка: {len(X_test)} записей ({len(X_test)/len(X)*100:.0f}%)")

📊 Обучающая выборка: 1149 записей (70%)
📊 Тестовая выборка: 493 записей (30%)


# базовая

In [121]:
cat_features_indices = [X_train.columns.get_loc(col) for col in categorical_features]

model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3,
    random_seed=42,
    loss_function='RMSE',
    eval_metric='RMSE',
    early_stopping_rounds=50,
    verbose=50,
    cat_features=cat_features_indices
)


print(" Начинаем обучение...\n")
model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    verbose=50,
    plot=False
)
print("\n Обучение завершено!")

 Начинаем обучение...

0:	learn: 0.9489542	test: 1.0254616	best: 1.0254616 (0)	total: 47.2ms	remaining: 23.5s
50:	learn: 0.4245245	test: 0.5189263	best: 0.5189263 (50)	total: 1.44s	remaining: 12.6s
100:	learn: 0.3555996	test: 0.4558640	best: 0.4558640 (100)	total: 2.79s	remaining: 11s
150:	learn: 0.3221567	test: 0.4314227	best: 0.4314227 (150)	total: 4.1s	remaining: 9.48s
200:	learn: 0.2981460	test: 0.4170647	best: 0.4170647 (200)	total: 5.38s	remaining: 8s
250:	learn: 0.2760731	test: 0.4084999	best: 0.4084645 (249)	total: 6.9s	remaining: 6.84s
300:	learn: 0.2598643	test: 0.4033871	best: 0.4033871 (300)	total: 8.38s	remaining: 5.54s
350:	learn: 0.2450507	test: 0.4004489	best: 0.4004489 (350)	total: 9.67s	remaining: 4.1s
400:	learn: 0.2337235	test: 0.3982162	best: 0.3982162 (400)	total: 11s	remaining: 2.72s
450:	learn: 0.2241056	test: 0.3955923	best: 0.3955891 (449)	total: 12.4s	remaining: 1.34s
499:	learn: 0.2141308	test: 0.3943079	best: 0.3943079 (499)	total: 13.6s	remaining: 0us

bes

In [122]:
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

In [123]:
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [124]:
print(f"\n   MAE (средняя ошибка): {mae:,.0f} ₽")
print(f"   RMSE: {rmse:,.0f} ₽")
print(f"   R² (качество модели): {r2:.4f}")
print(f"   MAPE (ошибка в %): {mape:.2f}%")


   MAE (средняя ошибка): 34,653,060 ₽
   RMSE: 123,596,922 ₽
   R² (качество модели): 0.5461
   MAPE (ошибка в %): 30.82%


In [125]:
importance = model.get_feature_importance()
importance_df = pd.DataFrame({
    'Признак': numeric_features + categorical_features,
    'Важность': importance
}).sort_values('Важность', ascending=False)

print("\nТоп-10 самых важных признаков:\n")
for i, row in importance_df.head(10).iterrows():
    print(f"   {i+1}. {row['Признак']}: {row['Важность']:.2f}")


Топ-10 самых важных признаков:

   1. total_area: 51.47
   4. total_floors: 8.84
   2. area_living: 8.43
   14. renovation: 5.19
   16. apartment_type: 4.54
   7. desc_length: 4.15
   12. house_type: 3.89
   13. bathroom: 3.78
   5. build_year: 2.93
   3. floor: 1.76


In [126]:
from datetime import datetime

model_filename = f"catboost_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}.cbm"
model.save_model(model_filename)
print(f" Модель сохранена: {model_filename}")


import pickle
with open('model_features.pkl', 'wb') as f:
    pickle.dump({
        'numeric_features': numeric_features,
        'categorical_features': categorical_features,
        'model_filename': model_filename,
        'metrics': {
            'mae': mae,
            'rmse': rmse,
            'r2': r2,
            'mape': mape
        }
    }, f)
print(f" Список признаков сохранен: model_features.pkl")

 Модель сохранена: catboost_model_20260428_211010.cbm
 Список признаков сохранен: model_features.pkl


# оптимизированная

In [128]:
if 'price_per_m2' in df.columns:
    df['price_per_m2_log'] = np.log1p(df['price_per_m2'])

df['living_ratio'] = df['area_living'] / df['total_area']
df['living_ratio'] = df['living_ratio'].clip(0.3, 0.95)

df['floor_pct'] = df['floor'] / df['total_floors']
df['floor_pct'] = df['floor_pct'].clip(0, 1)


current_year = 2026
df['house_age'] = current_year - df['build_year']
df['house_age'] = df['house_age'].clip(0, 100)


df['total_area_sq'] = df['total_area'] ** 2
df['floor_sq'] = df['floor'] ** 2


df['area_floor_interaction'] = df['total_area'] * df['floor_pct']


df['ceiling_category'] = pd.cut(df['ceiling_height'], 
                                  bins=[0, 2.5, 2.7, 3.0, 10], 
                                  labels=['low', 'standard', 'high', 'premium'])

In [129]:
target = 'price'

numeric_features = [
    'total_area', 'total_area_sq',       
    'area_living', 'living_ratio',        
    'floor', 'floor_sq', 'total_floors', 'floor_pct', 
    'build_year', 'house_age',            
    'ceiling_height',                      
    'images_count', 'desc_length',          
    'has_renovation', 'has_parking', 'has_balcony', 
    'area_floor_interaction']


numeric_features = [col for col in numeric_features if col in df.columns]

categorical_features = [
    'rooms', 'house_type', 'bathroom', 'renovation',
    'floor_category', 'apartment_type', 'ceiling_category'
]
categorical_features = [col for col in categorical_features if col in df.columns]

print(f"Числовых признаков: {len(numeric_features)}")
print(f"Категориальных признаков: {len(categorical_features)}")
print(f"\nСписок признаков:")
for f in numeric_features:
    print(f"   * {f}")
for f in categorical_features:
    print(f"   - {f}")

Числовых признаков: 17
Категориальных признаков: 7

Список признаков:
   * total_area
   * total_area_sq
   * area_living
   * living_ratio
   * floor
   * floor_sq
   * total_floors
   * floor_pct
   * build_year
   * house_age
   * ceiling_height
   * images_count
   * desc_length
   * has_renovation
   * has_parking
   * has_balcony
   * area_floor_interaction
   - rooms
   - house_type
   - bathroom
   - renovation
   - floor_category
   - apartment_type
   - ceiling_category


In [130]:
X = df[numeric_features + categorical_features].copy()
y = df[target].copy()

y_log = np.log1p(y)

In [133]:
base_params = {
    'random_seed': 42,
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'early_stopping_rounds': 50,
    'verbose': False
}

best_score = float('inf')
best_params = None
results = []

for lr in [0.05, 0.07]:
    for depth in [6, 8]:
        for l2 in [3, 5]:
            
            params = {
                'iterations': 500,
                'learning_rate': lr,
                'depth': depth,
                'l2_leaf_reg': l2,
                **base_params
            }
            
            cv_data = Pool(X, y_log, cat_features=[X.columns.get_loc(col) for col in categorical_features])
            cv_scores = cv(
                cv_data,
                params,
                fold_count=5,
                plot=False,
                verbose=False
            )
            
            mean_rmse = cv_scores['test-RMSE-mean'].values[-1]
            
            results.append({
                'iterations': iterations,
                'learning_rate': lr,
                'depth': depth,
                'l2_leaf_reg': l2,
                'rmse': mean_rmse
            })
            
            if mean_rmse < best_score:
                best_score = mean_rmse
                best_params = params
                
            print(f"   iter={iterations}, lr={lr}, depth={depth}, l2={l2}  RMSE={mean_rmse:.4f}")

print(f"\nЛучшие параметры:")
for k, v in best_params.items():
    print(f"   {k}: {v}")
print(f"   Лучший RMSE: {best_score:.4f}")

Training on fold [0/5]

bestTest = 0.4086882972
bestIteration = 493

Training on fold [1/5]

bestTest = 0.4525026363
bestIteration = 499

Training on fold [2/5]

bestTest = 0.4152138953
bestIteration = 489

Training on fold [3/5]

bestTest = 0.3983581222
bestIteration = 494

Training on fold [4/5]

bestTest = 0.3987739397
bestIteration = 498

   iter=300, lr=0.05, depth=6, l2=3  RMSE=0.4149
Training on fold [0/5]

bestTest = 0.4088586781
bestIteration = 498

Training on fold [1/5]

bestTest = 0.4434933848
bestIteration = 499

Training on fold [2/5]

bestTest = 0.4398196571
bestIteration = 491

Training on fold [3/5]

bestTest = 0.4177474505
bestIteration = 499

Training on fold [4/5]

bestTest = 0.4029802046
bestIteration = 499

   iter=300, lr=0.05, depth=6, l2=5  RMSE=0.4226
Training on fold [0/5]

bestTest = 0.4321531049
bestIteration = 499

Training on fold [1/5]

bestTest = 0.4659396302
bestIteration = 478

Training on fold [2/5]

bestTest = 0.4540604471
bestIteration = 497

Train

In [134]:
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42, shuffle=True)

best_params['verbose'] = 100
best_params['cat_features'] = [X_train.columns.get_loc(col) for col in categorical_features]

model = CatBoostRegressor(**best_params)
model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    verbose=100,
    plot=False
)

0:	learn: 0.9485554	test: 1.0406004	best: 1.0406004 (0)	total: 56.5ms	remaining: 28.2s
100:	learn: 0.3542870	test: 0.4566701	best: 0.4566701 (100)	total: 3.19s	remaining: 12.6s
200:	learn: 0.2961307	test: 0.4196255	best: 0.4196255 (200)	total: 6.05s	remaining: 9.01s
300:	learn: 0.2586497	test: 0.4057499	best: 0.4057499 (300)	total: 8.98s	remaining: 5.94s
400:	learn: 0.2313284	test: 0.3979332	best: 0.3979332 (400)	total: 11.9s	remaining: 2.94s
499:	learn: 0.2083320	test: 0.3927139	best: 0.3927139 (499)	total: 14.8s	remaining: 0us

bestTest = 0.3927139433
bestIteration = 499



CatBoostRegressor(cat_features=[17, 18, 19, 20, 21, 22, 23], depth=6, early_stopping_rounds=50, eval_metric='RMSE', iterations=500, l2_leaf_reg=3, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

In [135]:
# Предсказания
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

# Метрики
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print("="*60)
print(" ОЦЕНКА МОДЕЛИ ПОСЛЕ ОПТИМИЗАЦИИ")
print("="*60)
print(f"\n   MAE: {mae:,.0f} ₽")
print(f"   RMSE: {rmse:,.0f} ₽")
print(f"   R²: {r2:.4f}")
print(f"   MAPE: {mape:.2f}%")

 ОЦЕНКА МОДЕЛИ ПОСЛЕ ОПТИМИЗАЦИИ

   MAE: 37,012,393 ₽
   RMSE: 138,148,193 ₽
   R²: 0.5444
   MAPE: 30.45%


In [137]:
from datetime import datetime
import pickle

model_filename = f"catboost_model_optimized1_{datetime.now().strftime('%Y%m%d_%H%M%S')}.cbm"
model.save_model(model_filename)
print(f" Модель сохранена: {model_filename}")

with open('model_features_optimized.pkl', 'wb') as f:
    pickle.dump({
        'numeric_features': numeric_features,
        'categorical_features': categorical_features,
        'categorical_values': {col: df[col].unique().tolist() for col in categorical_features}
    }, f)
print(f" Признаки сохранены: model_features_optimized.pkl")

 Модель сохранена: catboost_model_optimized1_20260428_215324.cbm
 Признаки сохранены: model_features_optimized.pkl


In [152]:
new_apartment = pd.DataFrame({
    'total_area': [75.0],
    'area_living': [52.5],
    'living_ratio': [0.70],
    'floor': [8],
    'total_floors': [25],
    'floor_pct': [0.32],
    'build_year': [2022],
    'house_age': [4],
    'ceiling_height': [2.8],
    'total_area_sq': [5.625],
    'rooms': ['3'],
    'house_type': ['монолитный'],
    'bathroom': ['совмещенный'],
    'renovation': ['чистовая отделка'],
    'floor_category': ['low'],
    'apartment_type': ['medium'],
    'floor_group': ['low']
})

pred_log = model.predict(new_apartment)
pred_price = np.expm1(pred_log)[0]

print(f"\nПредсказанная цена: {pred_price:,.0f} ₽")
print(f"Диапазон: {pred_price * 0.8:,.0f} - {pred_price * 1.2:,.0f} ₽")


Предсказанная цена: 37,429,280 ₽
Диапазон: 29,943,424 - 44,915,136 ₽


# векторизация адреса

In [163]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import re

In [157]:
df = pd.read_csv(
    "realty_clean.csv", 
    encoding='utf-8-sig',
    sep=';',
    on_bad_lines='skip',
    low_memory=False
)

cols_to_drop = [col for col in df.columns if 'timestamp' in col or 'json_description' in col or 'latitude' in col or 'longitude' in col]
df = df.drop(columns=cols_to_drop, errors='ignore')

df = df.drop(columns='area_kitchen', errors='ignore')
df['area_living'] = df['area_living'].fillna(df['total_area'] * 0.7)
df['ceiling_height'] = df['ceiling_height'].fillna(2.7)

In [158]:
df.head()

,offer_id,title,price,total_area,area_living,rooms,floor,total_floors,address,description,...,images_count,s3_uris_count,s3_uris,price_per_m2,floor_category,apartment_type,desc_length,has_renovation,has_parking,has_balcony
0,3190558370547190502,"37,9 м², квартира-студия",30445648,37.9,15.60,студия,29.0,35.0,"Москва, ЖК Родина Парк, к1, ЖК «Родина Парк»",Разместить объявление Моя подборка ЖК Войти Но...,...,15.0,15.0,s3://realty-images/offers/3190558370547190502/...,8.033153e+05,high,studio,10293,1,1,0
1,5180127926972929670,"66,4 м², 3-комнатная квартира",35000000,66.4,15.00,3,16.0,16.0,"Москва, Небесный бульвар, 1к1, Жилой район «Алиа»",Разместить объявление Моя подборка ЖК Войти Но...,...,15.0,15.0,s3://realty-images/offers/5180127926972929670/...,5.271084e+05,high,medium,10219,1,1,0
2,2159627191506509600,"71,5 м², 2-комнатная квартира",102118256,71.5,50.05,2,14.0,18.0,"Москва, Украинский бульвар, 2с1, ЖК «Бадаевский»",Разместить объявление Моя подборка ЖК Войти Но...,...,15.0,15.0,s3://realty-images/offers/2159627191506509600/...,1.428227e+06,high,small,9853,1,1,0
3,5228459389428247141,"138 м², 4-комнатная квартира",135000000,138.0,96.60,4,7.0,9.0,"Москва, Малая Бронная улица, 25, ✅ Характерист...",Разместить объявление Моя подборка ЖК Войти Но...,...,15.0,15.0,s3://realty-images/offers/5228459389428247141/...,9.782609e+05,high,medium,12273,1,1,1
4,7035113340557147293,"74,4 м², 3-комнатная квартира",22145000,74.4,10.20,3,2.0,17.0,"Москва, улица Барышиха, 50, ✅ Характеристики, ...",Разместить объявление Моя подборка ЖК Войти Но...,...,15.0,15.0,s3://realty-images/offers/7035113340557147293/...,2.976478e+05,low,medium,13622,1,1,1


In [173]:
def clean_address(address):
    if pd.isna(address):
        return ''
    
    address = str(address)
    if '✅' in address:
        address = address.split('✅')[0]
    elif 'Характеристики' in address:
        address = address.split('Характеристики')[0]
    
    address = re.sub(r'<[^>]+>', ' ', address)
    address = re.sub(r'\s+', ' ', address).strip()
    garbage = ['as image', 'avatars', 'fetchpriority', 'high', 'imagesrcset', 
               'https', 'link', 'name description', 'description', 'null', 'undefined']
    for word in garbage:
        address = address.replace(word, '')
    address = re.sub(r'\s+', ' ', address).strip()
    
    return address

df['address_clean'] = df['address'].apply(clean_address)

df = df[df['address_clean'].str.len() > 10]

print(f"После очистки осталось {len(df)} записей")
print(f"\nПримеры очищенных адресов:")
for i in range(min(5, len(df))):
    print(f"   {i+1}. {df['address_clean'].iloc[i]}")

После очистки осталось 1641 записей

Примеры очищенных адресов:
   1. Москва, ЖК Родина Парк, к1, ЖК «Родина Парк»
   2. Москва, Небесный бульвар, 1к1, Жилой район «Алиа»
   3. Москва, Украинский бульвар, 2с1, ЖК «Бадаевский»
   4. Москва, Малая Бронная улица, 25,
   5. Москва, улица Барышиха, 50,


In [175]:
tfidf = TfidfVectorizer(
    max_features=100,         
    min_df=3,                 
    ngram_range=(1, 2),       
    analyzer='word',
    stop_words=None
)

address_tfidf = tfidf.fit_transform(df['address_clean'])

print(f" TF-IDF матрица: {address_tfidf.shape}")

feature_names = tfidf.get_feature_names_out()
print(f"\nТоп-30 значимых слов в адресах:")
for i, word in enumerate(feature_names[:30]):
    if len(word) > 2 and not word.isdigit():
        print(f"   {i+1}. {word}")

 TF-IDF матрица: (1641, 100)

Топ-30 значимых слов в адресах:
   4. 14с3
   5. 14с3 мфк
   7. 2с1
   8. 2с1 жк
   9. park
   10. park residences
   11. residences
   12. towers
   13. victory
   14. victory park
   15. академика
   16. апарт
   17. апарт комплекс
   18. бадаевский
   19. большая
   20. братьев
   21. братьев фонченко
   22. бульвар
   23. бульвар 2с1
   24. вал
   25. внуково
   26. город
   27. дау
   28. дом
   29. дом дау
   30. жилой


In [176]:
numeric_features = ['total_area', 'floor', 'total_floors', 'build_year', 'ceiling_height', 'images_count']

if 'area_living' in df.columns:
    numeric_features.append('area_living')
    df['area_living'] = df['area_living'].fillna(df['total_area'] * 0.7)

for col in numeric_features:
    if col in df.columns and df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

if 'ceiling_height' in df.columns:
    df['ceiling_height'] = df['ceiling_height'].fillna(2.7)

categorical_features = ['rooms', 'house_type', 'bathroom', 'renovation', 'floor_category', 'apartment_type']

In [182]:
categorical_features = ['rooms', 'house_type', 'bathroom', 'renovation', 'floor_category', 'apartment_type']

for col in categorical_features:
    if col in df.columns:

        df[col] = df[col].fillna('unknown')
        df[col] = df[col].astype(str)
        df[col] = df[col].replace('nan', 'unknown')

In [183]:
X_base = df[numeric_features + categorical_features].copy()

tfidf_df = pd.DataFrame(
    address_tfidf.toarray(),
    columns=[f'addr_{word}' for word in feature_names],
    index=X_base.index
)

X = pd.concat([X_base, tfidf_df], axis=1)

y = df['price'].copy()
y_log = np.log1p(y)
for col in categorical_features:
    if col in df.columns:
        df[col] = df[col].fillna('unknown').astype(str)

print(f"Размер X: {X.shape}")
print(f"Обычных признаков: {len(numeric_features) + len(categorical_features)}")
print(f"TF-IDF признаков: {len(feature_names)}")

Размер X: (1641, 113)
Обычных признаков: 13
TF-IDF признаков: 100


In [184]:
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

cat_features_idx = [X_train.columns.get_loc(col) for col in categorical_features if col in X_train.columns]

model = CatBoostRegressor(
    cat_features=cat_features_idx,
    depth=6,
    learning_rate=0.05,
    iterations=500,
    l2_leaf_reg=3,
    random_seed=42,
    early_stopping_rounds=50,
    eval_metric='RMSE',
    verbose=100
)

model.fit(X_train, y_train, eval_set=(X_test, y_test), verbose=100)

0:	learn: 0.9505864	test: 1.0342111	best: 1.0342111 (0)	total: 37.1ms	remaining: 18.5s
100:	learn: 0.3351547	test: 0.4381452	best: 0.4381452 (100)	total: 2.95s	remaining: 11.7s
200:	learn: 0.2710624	test: 0.3879649	best: 0.3879649 (200)	total: 6.03s	remaining: 8.97s
300:	learn: 0.2330797	test: 0.3661144	best: 0.3661144 (300)	total: 9.21s	remaining: 6.09s
400:	learn: 0.2095796	test: 0.3573938	best: 0.3573938 (400)	total: 12.8s	remaining: 3.17s
499:	learn: 0.1891578	test: 0.3494196	best: 0.3494196 (499)	total: 16s	remaining: 0us

bestTest = 0.3494196237
bestIteration = 499



CatBoostRegressor(cat_features=[7, 8, 9, 10, 11, 12], depth=6, early_stopping_rounds=50, eval_metric='RMSE', iterations=500, l2_leaf_reg=3, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

In [185]:
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f"\n MAE: {mae:,.0f} ₽")
print(f"RMSE: {rmse:,.0f} ₽")
print(f"R2: {r2:.4f}")
print(f"MAPE: {mape:.2f}%")



 MAE: 33,654,567 ₽
RMSE: 132,794,242 ₽
R2: 0.5690
MAPE: 23.99%


In [186]:
importance = model.get_feature_importance()
feature_names_full = X.columns.tolist()

importance_df = pd.DataFrame({
    'feature': feature_names_full,
    'importance': importance
}).sort_values('importance', ascending=False)

for i, row in importance_df.head(20).iterrows():
    marker = "!!" if row['feature'].startswith('addr_') else "   "
    print(f"{marker} {row['feature'][:35]}: {row['importance']:.2f}")

    total_area: 54.52
    area_living: 8.77
    total_floors: 4.65
!! addr_москва: 3.88
    renovation: 2.90
    apartment_type: 2.85
    bathroom: 2.22
    build_year: 1.65
    house_type: 1.44
    floor: 1.39
    rooms: 1.14
!! addr_park: 1.13
!! addr_жк: 1.07
!! addr_улица: 1.07
!! addr_клубный: 1.06
    ceiling_height: 0.93
!! addr_нао: 0.86
!! addr_дом: 0.84
!! addr_шоссе: 0.60
!! addr_москва нао: 0.57


In [189]:
results_df = pd.DataFrame({
    'address': df.loc[X_test.index[:10], 'address_clean'].values,
    'actual': y_true[:10],
    'predicted': y_pred[:10],
    'error_pct': ((y_true - y_pred) / y_true * 100).values[:10]
})

for i in range(len(results_df)):
    print(f"\n{results_df['address'].iloc[i][:50]}")
    print(f"Факт: {results_df['actual'].iloc[i]:,.0f} ")
    print(f"Предсказание: {results_df['predicted'].iloc[i]:,.0f} ")
    print(f"Ошибка: {results_df['error_pct'].iloc[i]:.1f}%")


Москва, НАО, Филимонковский район, ЖК Середневский
Факт: 10,767,900 
Предсказание: 11,440,073 
Ошибка: -6.2%

Москва, проспект Академика Сахарова, 7, ЖК RED7
Факт: 133,996,384 
Предсказание: 73,832,644 
Ошибка: 44.9%

Москва, Большая Садовая улица, 5к1, Квартал «Сады 
Факт: 124,800,000 
Предсказание: 159,296,417 
Ошибка: -27.6%

Москва, улица Намёткина, вл10, НАМЕТКИН ТАУЭР
Факт: 28,614,960 
Предсказание: 36,383,376 
Ошибка: -27.1%

Москва, Комсомольский проспект, 42с2, Лофт «Clerke
Факт: 100,000,000 
Предсказание: 29,788,005 
Ошибка: 70.2%

Москва, ЖК Репаблик, к2
Факт: 39,500,000 
Предсказание: 22,700,695 
Ошибка: 42.5%

Москва, улица Василисы Кожиной, 13, Квартал Match 
Факт: 39,943,900 
Предсказание: 44,279,902 
Ошибка: -10.9%

Москва, Автозаводская улица, 19к1,
Факт: 6,000,000 
Предсказание: 8,043,131 
Ошибка: -34.1%

Москва, улица Намёткина, 10Д, Апарт-отель AIST RES
Факт: 11,350,000 
Предсказание: 19,497,600 
Ошибка: -71.8%

Москва, 1-й Красногвардейский проезд, 13, ЖК «Оне»
Фа

In [209]:
if 'description' in df.columns:
    desc = df['description'].fillna('').astype(str)
    df['has_pool'] = desc.str.contains('бассейн', case=False).astype(int)
    df['has_fitness'] = desc.str.contains('фитнес|спортзал', case=False).astype(int)
    df['has_kids_room'] = desc.str.contains('детская|игровая', case=False).astype(int)
    df['has_parking'] = desc.str.contains('паркинг|подземный паркинг', case=False).astype(int)
    df['has_security'] = desc.str.contains('охрана|видеонаблюдение|консьерж', case=False).astype(int)
    df['has_terrace'] = desc.str.contains('терраса|балкон|лоджия', case=False).astype(int)


df['living_ratio'] = df['area_living'] / df['total_area']
df['living_ratio'] = df['living_ratio'].clip(0.3, 0.95).fillna(0.7)

df['floor_pct'] = df['floor'] / df['total_floors']
df['floor_pct'] = df['floor_pct'].clip(0, 1).fillna(0.5)

df['floor_category_detailed'] = pd.cut(df['floor_pct'], 
    bins=[0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0],
    labels=['ground', 'low', 'low_mid', 'mid', 'high_mid', 'top'])

current_year = 2026
df['house_age'] = current_year - df['build_year']
df['house_age'] = df['house_age'].clip(0, 100).fillna(10)

df['total_area_sq'] = df['total_area'] ** 2 / 1000
df['floor_sq'] = df['floor'] ** 2 / 100
#df['price_per_m2_log'] = np.log1p(df['price'] / df['total_area'])

df['area_floor_interaction'] = df['total_area'] * df['floor_pct']
df['area_age_interaction'] = df['total_area'] * df['house_age'] / 100

In [210]:
# Числовые признаки (расширенные)
numeric_features = [
    'total_area', 'area_living', 'floor', 'total_floors',
    'build_year', 'ceiling_height', 'images_count',
    'living_ratio', 'floor_pct', 'house_age',
    'total_area_sq', 'floor_sq',
    'area_floor_interaction', 'area_age_interaction',
    'has_pool', 'has_fitness', 'has_kids_room', 
    'has_parking', 'has_security', 'has_terrace'
]

numeric_features = [col for col in numeric_features if col in df.columns]

categorical_features = [
    'rooms', 'house_type', 'bathroom', 'renovation',
    'floor_category', 'apartment_type', 'district', 'floor_category_detailed'
]
categorical_features = [col for col in categorical_features if col in df.columns]

print(f"Числовых признаков: {len(numeric_features)}")
print(f"\nКатегориальных признаков: {len(categorical_features)}")

Числовых признаков: 20

Категориальных признаков: 8


In [211]:
tfidf = TfidfVectorizer(
    max_features=100,
    min_df=3,
    ngram_range=(1, 3) 
)
address_tfidf = tfidf.fit_transform(df['address_clean'].fillna(''))

X_base = df[numeric_features + categorical_features].copy()
tfidf_df = pd.DataFrame(
    address_tfidf.toarray(),
    columns=[f'addr_{word}' for word in tfidf.get_feature_names_out()],
    index=X_base.index
)
X = pd.concat([X_base, tfidf_df], axis=1)

y = df['price'].fillna(df['price'].median())
y_log = np.log1p(y)

print(f"X shape: {X.shape}")
print(f"Числовых: {len(numeric_features)}")
print(f"Категориальных: {len(categorical_features)}")
print(f"TF-IDF: {len(tfidf.get_feature_names_out())}")

X shape: (1641, 128)
Числовых: 20
Категориальных: 8
TF-IDF: 100


In [212]:
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42, shuffle=True)

cat_features_idx = [X_train.columns.get_loc(col) for col in categorical_features if col in X_train.columns]

model = CatBoostRegressor(
    cat_features=cat_features_idx,
    depth=8,                 
    learning_rate=0.07,       
    iterations=800,          
    l2_leaf_reg=2,            
    random_seed=42,
    early_stopping_rounds=50,
    eval_metric='RMSE',
    verbose=100,
    subsample=0.8,              
    colsample_bylevel=0.8  
)

model.fit(X_train, y_train, eval_set=(X_test, y_test), verbose=100)

0:	learn: 0.9403257	test: 1.0229814	best: 1.0229814 (0)	total: 55.3ms	remaining: 44.2s
100:	learn: 0.2557708	test: 0.3818149	best: 0.3818149 (100)	total: 5.52s	remaining: 38.2s
200:	learn: 0.1782507	test: 0.3459993	best: 0.3459993 (200)	total: 11.4s	remaining: 34s
300:	learn: 0.1392179	test: 0.3359513	best: 0.3359513 (300)	total: 17.4s	remaining: 28.9s
400:	learn: 0.1118127	test: 0.3307560	best: 0.3307442 (399)	total: 23.2s	remaining: 23.1s
500:	learn: 0.0900974	test: 0.3273252	best: 0.3271839 (497)	total: 28.9s	remaining: 17.3s
600:	learn: 0.0752573	test: 0.3250605	best: 0.3250546 (599)	total: 34.6s	remaining: 11.5s
700:	learn: 0.0645792	test: 0.3233499	best: 0.3233499 (700)	total: 40.4s	remaining: 5.71s
799:	learn: 0.0550494	test: 0.3226197	best: 0.3224221 (781)	total: 46.2s	remaining: 0us

bestTest = 0.3224220853
bestIteration = 781

Shrink model to first 782 iterations.


CatBoostRegressor(cat_features=[20, 21, 22, 23, 24, 25, 26, 27], colsample_bylevel=0.8, depth=8, early_stopping_rounds=50, eval_metric='RMSE', iterations=800, l2_leaf_reg=2, learning_rate=0.07, loss_function='RMSE', random_seed=42, subsample=0.8, verbose=100)

In [213]:
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f"MAE: {mae:,.0f} ₽")
print(f"RMSE: {rmse:,.0f} ₽")
print(f"R2: {r2:.4f}")
print(f"MAPE: {mape:.2f}%")

MAE: 32,911,211 ₽
RMSE: 126,807,553 ₽
R2: 0.6070
MAPE: 21.67%


In [214]:
importance = model.get_feature_importance()
importance_df = pd.DataFrame({
    'feature': X.columns.tolist(),
    'importance': importance
}).sort_values('importance', ascending=False)


for i, row in importance_df.head(25).iterrows():
    print(f"{row['feature'][:40]}: {row['importance']:.2f}")

total_area_sq: 25.72
total_area: 20.19
renovation: 5.11
area_living: 5.04
total_floors: 4.37
addr_москва: 3.39
district: 3.07
apartment_type: 3.01
area_floor_interaction: 2.49
bathroom: 1.88
house_type: 1.83
living_ratio: 1.63
rooms: 1.55
addr_жк: 1.28
addr_дом: 1.20
addr_park: 1.11
area_age_interaction: 1.03
addr_улица: 1.01
ceiling_height: 0.97
floor_pct: 0.89
floor_category_detailed: 0.84
has_fitness: 0.76
build_year: 0.69
floor_sq: 0.68
addr_москва нао: 0.60


In [215]:
model_filename = f"catboost_address_{datetime.now().strftime('%Y%m%d_%H%M%S')}.cbm"
model.save_model(model_filename)
print(f" Модель сохранена: {model_filename}")

importance_df.to_csv('feature_importance_final.csv', index=False)
print(f" Важность признаков: feature_importance_address.csv")


print(f"""
   R2 = {r2:.4f}
   MAPE = {mape:.2f}% 
""")

 Модель сохранена: catboost_address_20260428_231054.cbm
 Важность признаков: feature_importance_address.csv

   R2 = 0.6070
   MAPE = 21.67% 



*Добавлено признаков*:
   - Признаки из описания (бассейн, фитнес, паркинг, охрана)
   - Типы дома (кирпичный, монолитный, панельный)
   - Качество ремонта
   - Квадратичные и полиномиальные признаки
   - Взаимодействия (площадь×этаж, площадь×возраст)

#  присоединение описания

In [262]:
def clean_description(text):
    if pd.isna(text):
        return ''
    
    text = str(text)
    
    if 'Описание' in text:
        text = text.split('Описание')[-1]
    
    garbage_patterns = [
        r'Разместить объявление.*?Узнать больше',
        r'Недвижимость в Москве.*?Показать телефон',
        r'Журнал · Главное.*?Люди также ищут',
        r'Люди также ищут.*?$',
        r'Похожие объявления.*?$',
        r'Ранее вы смотрели.*?$',
        r'Ипотечный калькулятор.*?$',
        r'Динамика изменения цены.*?$',
        r'Собрали для вас лучшие рассрочки.*?$',
        r'Застройщик.*?\n',
        r'Показать телефон.*?\n',
        r'Добавить заметку.*?\n',
        r'Уточните у Недвижимость AI.*?\n',
    ]
    
    for pattern in garbage_patterns:
        text = re.sub(pattern, ' ', text, flags=re.DOTALL | re.IGNORECASE)
    
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\b\w{1,2}\b', ' ', text)
    text = text.lower()
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^а-я\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text[:3000]  

df['description_clean'] = df['description'].apply(clean_description)

print(f"\nПример очищенного description:")
for i in range(min(3, len(df))):
    print(f"\n   {i+1}. {df['description_clean'].iloc[i][:300]}...")


Пример очищенного description:

   1. жилом кластере родина парк рядом природным заказником долина реки сетунь прода тся квартира студия продуманная европланировка которую легко зонировать под ваши запросы потолки родина парк премиальный семейный кластер образовательным медиакластером строящийся западе москвы окружении природного запове...

   2. евро тр хкомнатная квартира роскошными видами прода тся самая успешная планировке евро тр хкомнатная квартира площадью последнем этаже современного дома бизнес класса главное достоинство роскошные виды москву обеих сторон восточные окна встречают рассветы западные дарят атмосферу закатов функциональ...

   3. прода тся комнатная квартира общей площадью этаже жилого комплекса бадаевский нем говорили его ждали жилой комплекс бадаевский кутузовском проспекте это место для жизни которое займет место истории девелопер бадаевском исторические здания завода футуристические парящие дома создают единый комплекс к...


In [217]:
numeric_features = [
    'total_area', 'area_living', 'floor', 'total_floors',
    'build_year', 'ceiling_height', 'images_count'
]

df['living_ratio'] = (df['area_living'] / df['total_area']).clip(0.3, 0.95).fillna(0.7)
df['floor_pct'] = (df['floor'] / df['total_floors']).clip(0, 1).fillna(0.5)
current_year = 2026
df['house_age'] = (current_year - df['build_year']).clip(0, 100).fillna(10)
df['total_area_sq'] = df['total_area'] ** 2 / 1000

In [218]:
tfidf_desc = TfidfVectorizer(
    max_features=200, 
    min_df=3,
    ngram_range=(1, 3),
    stop_words=['квартира', 'комнатная', 'этаж', 'дом', 'жилой', 'комплекс', 'продается']
)

desc_tfidf = tfidf_desc.fit_transform(df['description_clean'].fillna(''))
print(f" Description TF-IDF: {desc_tfidf.shape}")

tfidf_addr = TfidfVectorizer(
    max_features=50,
    min_df=3,
    ngram_range=(1, 2)
)

addr_tfidf = tfidf_addr.fit_transform(df['address_clean'].fillna(''))
print(f"Address TF-IDF: {addr_tfidf.shape}")

desc_words = tfidf_desc.get_feature_names_out()
print(f"\nТоп-20 слов в description:")
for i, word in enumerate(desc_words[:20]):
    print(f"   {i+1}. {word}")

 Description TF-IDF: (1641, 200)
Address TF-IDF: (1641, 50)

Топ-20 слов в description:
   1. агентство
   2. без
   3. бизнес
   4. будет
   5. бульвар
   6. бюджет
   7. бюджет вас
   8. бюджет вас пожелания
   9. варианты
   10. варианты оставить
   11. варианты оставить заявку
   12. вас
   13. вас пожелания
   14. вас пожелания нас
   15. ваш
   16. ваш бюджет
   17. ваш бюджет вас
   18. все
   19. всего
   20. вский


In [219]:
X_base = pd.DataFrame(index=df.index)

for col in numeric_features:
    if col in df.columns:
        X_base[col] = df[col].values

X_base['living_ratio'] = df['living_ratio'].values
X_base['floor_pct'] = df['floor_pct'].values
X_base['house_age'] = df['house_age'].values
X_base['total_area_sq'] = df['total_area_sq'].values

for col in categorical_features:
    if col in df.columns:
        X_base[col] = df[col].values

desc_df = pd.DataFrame(
    desc_tfidf.toarray(),
    columns=[f'desc_{word}' for word in desc_words],
    index=df.index
)

addr_df = pd.DataFrame(
    addr_tfidf.toarray(),
    columns=[f'addr_{word}' for word in tfidf_addr.get_feature_names_out()],
    index=df.index
)

X = pd.concat([X_base, desc_df, addr_df], axis=1)

y = df['price'].fillna(df['price'].median())
y_log = np.log1p(y)

print(f"\nИтоговый размер X: {X.shape}")
print(f"Числовых: {len(numeric_features) + 4}")
print(f"Категориальных: {len(categorical_features)}")
print(f"Description TF-IDF: {len(desc_words)}")
print(f"Address TF-IDF: {len(tfidf_addr.get_feature_names_out())}")


Итоговый размер X: (1641, 269)
Числовых: 11
Категориальных: 8
Description TF-IDF: 200
Address TF-IDF: 50


In [220]:
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42, shuffle=True)

cat_features_idx = [X_train.columns.get_loc(col) for col in categorical_features if col in X_train.columns]

print(f"Категориальных признаков в модели: {len(cat_features_idx)}")

model = CatBoostRegressor(
    cat_features=cat_features_idx,
    depth=8,
    learning_rate=0.07,
    iterations=800,
    l2_leaf_reg=3,
    random_seed=42,
    early_stopping_rounds=50,
    eval_metric='RMSE',
    verbose=100
)

model.fit(X_train, y_train, eval_set=(X_test, y_test), verbose=100)

Категориальных признаков в модели: 8
0:	learn: 0.9453223	test: 1.0292890	best: 1.0292890 (0)	total: 377ms	remaining: 5m 1s
100:	learn: 0.2086218	test: 0.3380967	best: 0.3380967 (100)	total: 24.5s	remaining: 2m 49s
200:	learn: 0.1284455	test: 0.3012563	best: 0.3012563 (200)	total: 48.1s	remaining: 2m 23s
300:	learn: 0.0936394	test: 0.2937474	best: 0.2937474 (300)	total: 1m 12s	remaining: 1m 59s
400:	learn: 0.0732403	test: 0.2909502	best: 0.2909502 (400)	total: 1m 35s	remaining: 1m 35s
500:	learn: 0.0572129	test: 0.2882912	best: 0.2882912 (500)	total: 2m	remaining: 1m 11s
600:	learn: 0.0465003	test: 0.2875031	best: 0.2874081 (594)	total: 2m 23s	remaining: 47.6s
700:	learn: 0.0386377	test: 0.2865826	best: 0.2865826 (700)	total: 2m 47s	remaining: 23.6s
799:	learn: 0.0321492	test: 0.2861424	best: 0.2861376 (795)	total: 3m 11s	remaining: 0us

bestTest = 0.2861375957
bestIteration = 795

Shrink model to first 796 iterations.


CatBoostRegressor(cat_features=[11, 12, 13, 14, 15, 16, 17, 18], depth=8, early_stopping_rounds=50, eval_metric='RMSE', iterations=800, l2_leaf_reg=3, learning_rate=0.07, loss_function='RMSE', random_seed=42, verbose=100)

In [221]:
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f"\nMAE: {mae:,.0f} ₽")
print(f"RMSE: {rmse:,.0f} ₽")
print(f"R2: {r2:.4f}")
print(f"MAPE: {mape:.2f}%")


MAE: 30,802,598 ₽
RMSE: 133,558,423 ₽
R2: 0.5640
MAPE: 18.12%


In [222]:
model_filename = f"catboost_nlp_{datetime.now().strftime('%Y%m%d_%H%M%S')}.cbm"
model.save_model(model_filename)
print(f"Модель сохранена: {model_filename}")

importance_df = pd.DataFrame({
    'feature': X.columns.tolist(),
    'importance': model.get_feature_importance()
}).sort_values('importance', ascending=False)

importance_df.to_csv('feature_importance_final.csv', index=False)
print(f"Важность признаков: feature_importance_nlp.csv")

Модель сохранена: catboost_nlp_20260428_233302.cbm
Важность признаков: feature_importance_nlp.csv


# адрес + описание+ признаки

In [223]:

df = pd.read_csv(
    "realty_clean.csv", 
    encoding='utf-8-sig',
    sep=';',
    on_bad_lines='skip',
    low_memory=False
)


def clean_description(text):
    if pd.isna(text):
        return ''
    text = str(text)
    if 'Описание' in text:
        text = text.split('Описание')[-1]
    text = re.sub(r'Разместить объявление.*?Узнать больше', ' ', text, flags=re.DOTALL)
    text = re.sub(r'Недвижимость в Москве.*?Показать телефон', ' ', text, flags=re.DOTALL)
    text = re.sub(r'Журнал.*?Люди также ищут', ' ', text, flags=re.DOTALL)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.lower()
    text = re.sub(r'[^а-я\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text[:3000]

df['description_clean'] = df['description'].apply(clean_description)

def clean_address(address):
    if pd.isna(address):
        return ''
    address = str(address)
    if '✅' in address:
        address = address.split('✅')[0]
    elif 'Характеристики' in address:
        address = address.split('Характеристики')[0]
    address = re.sub(r'<[^>]+>', ' ', address)
    address = re.sub(r'\s+', ' ', address).strip()
    return address

df['address_clean'] = df['address'].apply(clean_address)

In [243]:
desc = df['description'].fillna('').astype(str).str.lower()

features_dict = {
    'has_pool': ['бассейн', 'spa', 'спа', 'аквазона'],
    'has_fitness': ['фитнес', 'спортзал', 'тренажерный', 'спортивный зал', 'фитнес-центр'],
    'has_security': ['охрана', 'видеонаблюдение', 'консьерж', 'безопасность', 'домофон', 'кпп'],
    'has_terrace': ['терраса', 'балкон', 'лоджия', 'веранда'],
    'has_kids_room': ['детская', 'игровая', 'детский сад', 'детская площадка'],
    'has_lift': ['лифт', 'скоростной лифт', 'грузовой лифт'],
    'has_conditioner': ['кондиционер', 'сплит-система', 'климат-контроль', 'вентиляция'],
    'has_workplace': ['коворкинг', 'рабочее место', 'переговорная', 'бизнес-центр'],
    'has_renovation_high': ['дизайнерский ремонт', 'премиум ремонт', 'евроремонт', 'под ключ'],
    'has_furniture': ['мебель', 'встроенная мебель', 'кухонный гарнитур', 'шкаф'],
    'has_internet': ['интернет', 'wi-fi', 'оптоволокно', 'связь'],
    'has_penthouse': ['пентхаус', 'верхний этаж', 'последний этаж'],
}

for name, keywords in features_dict.items():
    pattern = '|'.join(keywords)
    df[name] = desc.str.contains(pattern, na=False).astype(int)

print("\n Бинарные признаки (частота встречаемости):")
for name in features_dict.keys():
    count = df[name].sum()
    pct = count / len(df) * 100
    print(f"   {name}: {count} ({pct:.1f}%)")


 Бинарные признаки (частота встречаемости):
   has_pool: 947 (57.7%)
   has_fitness: 518 (31.5%)
   has_security: 1206 (73.4%)
   has_terrace: 681 (41.5%)
   has_kids_room: 501 (30.5%)
   has_lift: 1368 (83.3%)
   has_conditioner: 212 (12.9%)
   has_workplace: 369 (22.5%)
   has_renovation_high: 396 (24.1%)
   has_furniture: 398 (24.2%)
   has_internet: 161 (9.8%)
   has_penthouse: 191 (11.6%)


In [244]:
numeric_features = [
    'total_area', 'area_living', 'floor', 'total_floors',
    'build_year', 'ceiling_height', 'images_count'
]

for col in numeric_features:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].median() if not df[col].isna().all() else 0)

df['living_ratio'] = (df['area_living'] / df['total_area']).clip(0.3, 0.95).fillna(0.7)
df['floor_pct'] = (df['floor'] / df['total_floors']).clip(0, 1).fillna(0.5)
current_year = 2026
df['house_age'] = (current_year - df['build_year']).clip(0, 100).fillna(10)

all_numeric = numeric_features + ['living_ratio', 'floor_pct', 'house_age'] + list(features_dict.keys())

print(f"Числовых признаков: {len(all_numeric)}")

Числовых признаков: 22


In [256]:
categorical_features = ['rooms', 'house_type', 'bathroom', 'renovation', 'apartment_type']
print(f" Категориальных признаков: {len(categorical_features)}")

 Категориальных признаков: 5


In [257]:
tfidf_desc = TfidfVectorizer(
    max_features=150, 
    min_df=3,
    ngram_range=(1, 2)
)

desc_tfidf = tfidf_desc.fit_transform(df['description_clean'].fillna(''))
print(f"Description TF-IDF: {desc_tfidf.shape}")

tfidf_addr = TfidfVectorizer(
    max_features=50,
    min_df=3,
    ngram_range=(1, 2)
)

addr_tfidf = tfidf_addr.fit_transform(df['address_clean'].fillna(''))
print(f"Address TF-IDF: {addr_tfidf.shape}")

Description TF-IDF: (1642, 150)
Address TF-IDF: (1642, 50)


In [258]:
X_base = pd.DataFrame(index=df.index)

for col in all_numeric:
    if col in df.columns:
        X_base[col] = df[col].values

for col in categorical_features:
    if col in df.columns:
        X_base[col] = df[col].values

desc_df = pd.DataFrame(
    desc_tfidf.toarray(),
    columns=[f'desc_{word}' for word in tfidf_desc.get_feature_names_out()],
    index=df.index
)

addr_df = pd.DataFrame(
    addr_tfidf.toarray(),
    columns=[f'addr_{word}' for word in tfidf_addr.get_feature_names_out()],
    index=df.index
)

X = pd.concat([X_base, desc_df, addr_df], axis=1)
X = X.fillna(0)
y = df['price'].fillna(df['price'].median())
y_log = np.log1p(y)

print(f"\nИтоговый размер X: {X.shape}")
print(f"Числовых (включая бинарные): {len(all_numeric)}")
print(f"Категориальных: {len(categorical_features)}")
print(f"TF-IDF: {desc_tfidf.shape[1] + addr_tfidf.shape[1]}")


Итоговый размер X: (1642, 227)
Числовых (включая бинарные): 22
Категориальных: 5
TF-IDF: 200


In [259]:
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42, shuffle=True)

cat_features_idx = [X_train.columns.get_loc(col) for col in categorical_features if col in X_train.columns]

model = CatBoostRegressor(
    cat_features=cat_features_idx,
    depth=7,
    learning_rate=0.07,
    iterations=700,
    l2_leaf_reg=3,
    random_seed=42,
    early_stopping_rounds=50,
    eval_metric='RMSE',
    verbose=100
)

model.fit(X_train, y_train, eval_set=(X_test, y_test), verbose=100)

0:	learn: 0.9460173	test: 1.0388823	best: 1.0388823 (0)	total: 103ms	remaining: 1m 12s
100:	learn: 0.2267028	test: 0.3823207	best: 0.3823207 (100)	total: 11.2s	remaining: 1m 6s
200:	learn: 0.1394013	test: 0.3482315	best: 0.3482288 (199)	total: 22s	remaining: 54.6s
300:	learn: 0.0985007	test: 0.3379975	best: 0.3379975 (300)	total: 34.2s	remaining: 45.4s
400:	learn: 0.0716182	test: 0.3332989	best: 0.3332583 (398)	total: 45.1s	remaining: 33.7s
500:	learn: 0.0545628	test: 0.3311952	best: 0.3311383 (498)	total: 55.5s	remaining: 22s
600:	learn: 0.0416468	test: 0.3303154	best: 0.3303154 (600)	total: 1m 6s	remaining: 10.9s
699:	learn: 0.0333972	test: 0.3297454	best: 0.3297454 (699)	total: 1m 16s	remaining: 0us

bestTest = 0.3297454049
bestIteration = 699



CatBoostRegressor(cat_features=[22, 23, 24, 25, 26], depth=7, early_stopping_rounds=50, eval_metric='RMSE', iterations=700, l2_leaf_reg=3, learning_rate=0.07, loss_function='RMSE', random_seed=42, verbose=100)

In [261]:
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f"\n   MAE: {mae:,.0f} ₽")
print(f"   RMSE: {rmse:,.0f} ₽")
print(f"   R2: {r2:.4f}")
print(f"   MAPE: {mape:.2f}%")


   MAE: 34,943,254 ₽
   RMSE: 144,721,326 ₽
   R2: 0.5000
   MAPE: 21.92%


неудача

# улучшенная nlp

In [263]:
df = pd.read_csv(
    "realty_clean.csv", 
    encoding='utf-8-sig',
    sep=';',
    on_bad_lines='skip',
    low_memory=False
)

cols_to_drop = [col for col in df.columns if 'timestamp' in col or 'json_description' in col or 'latitude' in col or 'longitude' in col]
df = df.drop(columns=cols_to_drop, errors='ignore')

print(f"Загружено {len(df)} записей")

price_threshold = df['price'].quantile(0.99)
df = df[df['price'] <= price_threshold]
print(f"После удаления выбросов (>99 перцентиля): {len(df)} записей")

def clean_address(address):
    if pd.isna(address):
        return ''
    address = str(address)
    if '✅' in address:
        address = address.split('✅')[0]
    elif 'Характеристики' in address:
        address = address.split('Характеристики')[0]
    address = re.sub(r'<[^>]+>', ' ', address)
    address = re.sub(r'\s+', ' ', address).strip()
    return address

def clean_description(text):
    if pd.isna(text):
        return ''
    text = str(text)
    if 'Описание' in text:
        text = text.split('Описание')[-1]

    text = re.sub(r'Разместить объявление.*?Узнать больше', ' ', text, flags=re.DOTALL)
    text = re.sub(r'Недвижимость в Москве.*?Показать телефон', ' ', text, flags=re.DOTALL)
    text = re.sub(r'Журнал.*?Люди также ищут', ' ', text, flags=re.DOTALL)
    text = re.sub(r'Похожие объявления.*?$', ' ', text, flags=re.DOTALL)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.lower()
    text = re.sub(r'[^а-я\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text[:3000]

df['address_clean'] = df['address'].apply(clean_address)
df['description_clean'] = df['description'].apply(clean_description)
print("\nТексты очищены")

numeric_features = ['total_area', 'floor', 'total_floors', 'build_year', 'ceiling_height', 'images_count']
for col in numeric_features:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].median())


df['living_ratio'] = (df['area_living'] / df['total_area']).clip(0.3, 0.95).fillna(0.7)
df['floor_pct'] = (df['floor'] / df['total_floors']).clip(0, 1).fillna(0.5)
current_year = 2026
df['house_age'] = (current_year - df['build_year']).clip(0, 100).fillna(10)
df['total_area_sq'] = df['total_area'] ** 2 / 1000

all_numeric = numeric_features + ['living_ratio', 'floor_pct', 'house_age', 'total_area_sq']


categorical_features = ['rooms', 'house_type', 'bathroom', 'renovation', 'floor_category', 'apartment_type']
for col in categorical_features:
    if col in df.columns:
        df[col] = df[col].fillna('unknown').astype(str)

print(f"Признаки: числовых={len(all_numeric)}, категориальных={len(categorical_features)}")

tfidf_desc = TfidfVectorizer(
    max_features=150,
    min_df=3,
    ngram_range=(1, 2)
)
desc_tfidf = tfidf_desc.fit_transform(df['description_clean'].fillna(''))

tfidf_addr = TfidfVectorizer(
    max_features=30,
    min_df=3,
    ngram_range=(1, 2)
)
addr_tfidf = tfidf_addr.fit_transform(df['address_clean'].fillna(''))

print(f"Description TF-IDF: {desc_tfidf.shape[1]} признаков")
print(f"Address TF-IDF: {addr_tfidf.shape[1]} признаков")

X_base = pd.DataFrame(index=df.index)
for col in all_numeric:
    if col in df.columns:
        X_base[col] = df[col].values
for col in categorical_features:
    if col in df.columns:
        X_base[col] = df[col].values

desc_df = pd.DataFrame(
    desc_tfidf.toarray(),
    columns=[f'desc_{word}' for word in tfidf_desc.get_feature_names_out()],
    index=df.index
)
addr_df = pd.DataFrame(
    addr_tfidf.toarray(),
    columns=[f'addr_{word}' for word in tfidf_addr.get_feature_names_out()],
    index=df.index
)

X = pd.concat([X_base, desc_df, addr_df], axis=1).fillna(0)

y = df['price'].copy()
y_log = np.log1p(y)

print(f"X shape: {X.shape}")

X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

cat_features_idx = [X_train.columns.get_loc(col) for col in categorical_features if col in X_train.columns]

model = CatBoostRegressor(
    cat_features=cat_features_idx,
    depth=7,
    learning_rate=0.06,
    iterations=600,
    l2_leaf_reg=3,
    random_seed=42,
    early_stopping_rounds=50,
    eval_metric='RMSE',
    verbose=100
)

model.fit(X_train, y_train, eval_set=(X_test, y_test), verbose=100)

y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f"\nMAE: {mae:,.0f} ₽")
print(f"RMSE: {rmse:,.0f} ₽")
print(f"R2: {r2:.4f}")
print(f"MAPE: {mape:.2f}%")

Загружено 1642 записей
После удаления выбросов (>99 перцентиля): 1625 записей

Тексты очищены
Признаки: числовых=10, категориальных=6
Description TF-IDF: 150 признаков
Address TF-IDF: 30 признаков
X shape: (1625, 196)
0:	learn: 0.9225800	test: 0.9216831	best: 0.9216831 (0)	total: 113ms	remaining: 1m 7s
100:	learn: 0.2401497	test: 0.3186655	best: 0.3186655 (100)	total: 10.7s	remaining: 52.8s
200:	learn: 0.1506794	test: 0.2824876	best: 0.2824876 (200)	total: 21.2s	remaining: 42s
300:	learn: 0.1051295	test: 0.2743032	best: 0.2743032 (300)	total: 31.7s	remaining: 31.5s
400:	learn: 0.0789657	test: 0.2715230	best: 0.2715230 (400)	total: 42.5s	remaining: 21.1s
500:	learn: 0.0617440	test: 0.2699650	best: 0.2699650 (500)	total: 53.9s	remaining: 10.7s
599:	learn: 0.0478954	test: 0.2688074	best: 0.2688074 (599)	total: 1m 4s	remaining: 0us

bestTest = 0.2688074399
bestIteration = 599


MAE: 17,216,158 ₽
RMSE: 44,713,766 ₽
R2: 0.7973
MAPE: 19.34%


In [264]:
best_nlp_model = model

In [265]:
best_nlp_model_filename = f"catboost_nlp_best_{datetime.now().strftime('%Y%m%d_%H%M%S')}.cbm"
best_nlp_model.save_model(best_nlp_model_filename)
print(f"Модель сохранена: {best_nlp_model_filename}")

Модель сохранена: catboost_nlp_best_20260429_002359.cbm


# эмбеддинг фотографий

In [266]:
!pip install torch torchvision transformers sentence-transformers pillow minio

  Using cached minio-7.2.20-py3-none-any.whl.metadata (6.5 kB)
  Using cached pycryptodome-3.23.0-cp37-abi3-win_amd64.whl.metadata (3.5 kB)
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/114.6 MB ? eta -:--:--
   ---------------------------------------- 1.3/114.6 MB 4.2 MB/s eta 0:00:28
    --------------------------------------- 2.1/114.6 MB 4.4 MB/s eta 0:00:26
    --------------------------------------- 2.6/114.6 MB 3.9 MB/s eta 0:00:29
   - -------------------------------------- 3.9/114.6 MB 4.2 MB/s eta 0:00:27
   - -------------------------------------- 5.0/114.6 MB 4.4 MB/s eta 0:00:26
   -- ------------------------------------- 5.8/114.6 MB 4.3 MB/s eta 0:00:26
   -- ------------------------------------- 6.6/114.6 MB 4.3 MB/s eta 0:00:26
   -- ------------------------------------- 7.6/114.6 MB 4.3 MB/s eta 0:00:25
   --- ------------------------------------ 8.7/114.6 MB 4.3 MB/s eta 0:00:25
   --- ---------

In [267]:
import os
import torch
from PIL import Image
from io import BytesIO
from minio import Minio
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer

In [269]:
minio_client = Minio(
    "localhost:9000",
    access_key="minioadmin",
    secret_key="minioadmin123",
    secure=False
)

bucket_name = "realty-images"

# Проверяем доступность
if minio_client.bucket_exists(bucket_name):
    print(f" Connected to MinIO bucket: {bucket_name}")
else:
    print(f" Bucket {bucket_name} not found")

 Connected to MinIO bucket: realty-images


In [270]:
model = SentenceTransformer('clip-ViT-B-32')

print(f"Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")

modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

0_CLIPModel/model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/604 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Model loaded. Embedding dimension: 512


In [272]:
def get_image_embedding_from_minio(offer_id, minio_client, bucket_name, model, max_images=15):
    embeddings = []
    prefix = f"offers/{offer_id}/"
    
    try:
        objects = list(minio_client.list_objects(bucket_name, prefix=prefix, recursive=True))
        
        if not objects:
            return None
        
        objects = sorted(objects, key=lambda x: x.object_name)[:max_images]
        
        for obj in objects:
            try:
                response = minio_client.get_object(bucket_name, obj.object_name)
                image_data = response.read()
                response.close()
                
                image = Image.open(BytesIO(image_data)).convert('RGB')
                
                embedding = model.encode(image)
                embeddings.append(embedding)
                
            except Exception as e:
                continue
        
        if not embeddings:
            return None
        
        return np.mean(embeddings, axis=0)
        
    except Exception as e:
        return None

test_offer_id = str(df['offer_id'].iloc[3])
print(f"Testing on offer_id: {test_offer_id}")
test_embedding = get_image_embedding_from_minio(test_offer_id, minio_client, bucket_name, model)

if test_embedding is not None:
    print(f"Got embedding with shape: {test_embedding.shape}")
else:
    print("No images found for this offer")

Testing on offer_id: 5228459389428247141
Got embedding with shape: (512,)


In [273]:
df = pd.read_csv(
    "realty_clean.csv", 
    encoding='utf-8-sig',
    sep=';',
    on_bad_lines='skip',
    low_memory=False
)

df['offer_id'] = df['offer_id'].astype(str)

print("\nGenerating image embeddings from MinIO...")

embeddings_list = []
success_count = 0

for idx, offer_id in tqdm(enumerate(df['offer_id']), total=len(df)):
    embedding = get_image_embedding_from_minio(offer_id, minio_client, bucket_name, model, max_images=10)
    
    if embedding is not None:
        embeddings_list.append(embedding)
        success_count += 1
    else:
        embeddings_list.append(np.zeros(model.get_sentence_embedding_dimension()))

print(f"\nSuccessfully got embeddings for {success_count}/{len(df)} offers")


Generating image embeddings from MinIO...


  0%|          | 0/1642 [00:00<?, ?it/s]


Successfully got embeddings for 1641/1642 offers


In [274]:
embedding_dim = model.get_sentence_embedding_dimension()
embedding_columns = [f'img_emb_{i}' for i in range(embedding_dim)]
embedding_df = pd.DataFrame(embeddings_list, columns=embedding_columns, index=df.index)

df_with_images = pd.concat([df, embedding_df], axis=1)

print(f"Shape with embeddings: {df_with_images.shape}")
print(f"New features: {len(embedding_columns)} image embedding columns")

Shape with embeddings: (1642, 540)
New features: 512 image embedding columns


In [275]:
exclude_cols = ['offer_id', 'title', 'address', 'description', 
                'description_clean', 'address_clean', 'url', 's3_uris', 'price']

numeric_features = ['total_area', 'area_living', 'floor', 'total_floors', 
                    'build_year', 'ceiling_height', 'images_count']

df_with_images['living_ratio'] = (df_with_images['area_living'] / df_with_images['total_area']).clip(0.3, 0.95).fillna(0.7)
df_with_images['floor_pct'] = (df_with_images['floor'] / df_with_images['total_floors']).clip(0, 1).fillna(0.5)
current_year = 2026
df_with_images['house_age'] = (current_year - df_with_images['build_year']).clip(0, 100).fillna(10)

all_numeric = numeric_features + ['living_ratio', 'floor_pct', 'house_age']


categorical_features = ['rooms', 'house_type', 'bathroom', 'renovation', 'floor_category', 'apartment_type']
for col in categorical_features:
    if col in df_with_images.columns:
        df_with_images[col] = df_with_images[col].fillna('unknown').astype(str)

image_features = [col for col in embedding_columns if col in df_with_images.columns]

print(f"Feature counts:")
print(f"Numeric: {len(all_numeric)}")
print(f"Categorical: {len(categorical_features)}")
print(f"Image embeddings: {len(image_features)}")

Feature counts:
Numeric: 10
Categorical: 6
Image embeddings: 512


In [276]:
X_base = df_with_images[all_numeric + categorical_features + image_features].copy()
y = df_with_images['price'].copy()
y_log = np.log1p(y)

X_train, X_test, y_train, y_test = train_test_split(X_base, y_log, test_size=0.2, random_state=42)
cat_features_idx = [X_train.columns.get_loc(col) for col in categorical_features if col in X_train.columns]

model_with_images = CatBoostRegressor(
    cat_features=cat_features_idx,
    depth=7,
    learning_rate=0.06,
    iterations=600,
    l2_leaf_reg=3,
    random_seed=42,
    early_stopping_rounds=50,
    eval_metric='RMSE',
    verbose=100
)

model_with_images.fit(X_train, y_train, eval_set=(X_test, y_test), verbose=100)

0:	learn: 0.9508238	test: 1.0430303	best: 1.0430303 (0)	total: 360ms	remaining: 3m 35s
100:	learn: 0.2828069	test: 0.4841755	best: 0.4841755 (100)	total: 29.8s	remaining: 2m 27s
200:	learn: 0.1678561	test: 0.4482035	best: 0.4482035 (200)	total: 58.3s	remaining: 1m 55s
300:	learn: 0.1029587	test: 0.4446963	best: 0.4445728 (298)	total: 1m 25s	remaining: 1m 24s
400:	learn: 0.0638156	test: 0.4442521	best: 0.4441527 (381)	total: 1m 54s	remaining: 56.9s
500:	learn: 0.0404930	test: 0.4435502	best: 0.4434601 (483)	total: 2m 23s	remaining: 28.4s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.443389489
bestIteration = 536

Shrink model to first 537 iterations.


CatBoostRegressor(cat_features=[10, 11, 12, 13, 14, 15], depth=7, early_stopping_rounds=50, eval_metric='RMSE', iterations=600, l2_leaf_reg=3, learning_rate=0.06, loss_function='RMSE', random_seed=42, verbose=100)

In [278]:
y_pred_log = model_with_images.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

mae_img = mean_absolute_error(y_true, y_pred)
rmse_img = np.sqrt(mean_squared_error(y_true, y_pred))
r2_img = r2_score(y_true, y_pred)
mape_img = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f"MAE: {mae_img:,.0f} ₽")
print(f"RMSE: {rmse_img:,.0f} ₽")
print(f"R2: {r2_img:.4f}")
print(f"MAPE: {mape_img:.2f}%")

MAE: 44,922,654 ₽
RMSE: 163,976,385 ₽
R2: 0.3581
MAPE: 31.85%


In [286]:
from sklearn.decomposition import PCA

In [297]:
X_indices = X.index.tolist()
y_filtered = y.loc[X_indices]
y_log_filtered = y_log.loc[X_indices]

embeddings_filtered = np.array([embeddings_list[i] for i in X_indices])


pca = PCA(n_components=32, random_state=42)
embeddings_32 = pca.fit_transform(embeddings_filtered)

old_emb_cols = [col for col in X.columns if col.startswith('img_emb_')]
if old_emb_cols:
    X = X.drop(columns=old_emb_cols)

for i in range(32):
    X[f'img_emb_pca_{i}'] = embeddings_32[:, i]

In [298]:
X_train, X_test, y_train, y_test = train_test_split(X, y_log_filtered, test_size=0.2, random_state=42)
print(f"📊 Train: {X_train.shape}, Test: {X_test.shape}")


cat_features_idx = [X_train.columns.get_loc(col) for col in categorical_features if col in X_train.columns]

model_fast = CatBoostRegressor(
    cat_features=cat_features_idx,
    depth=7,
    learning_rate=0.06,
    iterations=500,
    l2_leaf_reg=3,
    random_seed=42,
    early_stopping_rounds=100,
    verbose=50
)

model_fast.fit(X_train, y_train, eval_set=(X_test, y_test), verbose=50)

📊 Train: (1300, 228), Test: (325, 228)
0:	learn: 0.9234694	test: 0.9195892	best: 0.9195892 (0)	total: 152ms	remaining: 45.3s
50:	learn: 0.3363012	test: 0.3843624	best: 0.3843624 (50)	total: 6.96s	remaining: 34s
100:	learn: 0.2410260	test: 0.3220768	best: 0.3220768 (100)	total: 14.3s	remaining: 28.1s
150:	learn: 0.1888797	test: 0.3008118	best: 0.3008118 (150)	total: 22.8s	remaining: 22.5s
200:	learn: 0.1483121	test: 0.2914737	best: 0.2912159 (196)	total: 29.8s	remaining: 14.7s
250:	learn: 0.1206473	test: 0.2876954	best: 0.2876954 (250)	total: 36.3s	remaining: 7.09s
299:	learn: 0.0999835	test: 0.2836951	best: 0.2836951 (299)	total: 42.6s	remaining: 0us

bestTest = 0.2836951217
bestIteration = 299



CatBoostRegressor(cat_features=[10, 11, 12, 13, 14, 15], depth=7, early_stopping_rounds=50, iterations=300, l2_leaf_reg=3, learning_rate=0.06, loss_function='RMSE', random_seed=42, verbose=50)

In [299]:
y_pred_log = model_fast.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

mape_new = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
r2_new = r2_score(y_true, y_pred)

In [302]:
mae_new = mean_absolute_error(y_true, y_pred)
rmse_new = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"MAE: {mae_new:,.0f} ₽")
print(f"RMSE: {rmse_new:,.0f} ₽")
print(f"R2: {r2_new:.4f}")
print(f"MAPE: {mape_new:.2f}%")

MAE: 18,203,475 ₽
RMSE: 45,413,044 ₽
R2: 0.7909
MAPE: 20.89%


In [304]:
EMBEDDINGS_PATH = "embeddings.pkl" 

In [305]:
offer_ids_filtered = df.loc[X_indices, 'offer_id'].values

embeddings_df = pd.DataFrame({
    'offer_id': offer_ids_filtered,
    'clip_embedding': list(embeddings_32)
})

print(f"Подготовлено {len(embeddings_df)} эмбеддингов")
print(f"Размер эмбеддинга: {len(embeddings_df['clip_embedding'].iloc[0])}")

with open(EMBEDDINGS_PATH, 'wb') as f:
    pickle.dump(embeddings_df, f)

print(f"Сохранено в {EMBEDDINGS_PATH}")

Подготовлено 1625 эмбеддингов
Размер эмбеддинга: 32
Сохранено в embeddings.pkl


In [306]:
best_with_images = f"best_with_images.cbm"
model_fast.save_model(best_with_images)
print(f"Модель сохранена: {best_with_images}")

Модель сохранена: best_with_images.cbm
